# Phase A3: Type Head Training (Linear 256→5)

**Goal:** Train lightweight MLP type head on top of SAM3+LoRA (Phase A2) để output
per-detection per-class probability $p_{ik}$ — input cho Phase C/D conformal counting.

**Architecture:**
- SAM3 backbone (frozen) → multi-scale FPN features
- SAM3 decoder + LoRA (frozen, từ A2) → N detections {mask_i, score_i}
- For each detection: ROI-pool backbone features under mask_i → 256-d feature
- **TypeHead** Linear(256 → 128 → 5) → cross-entropy loss

**Training:**
- Generic prompt 'cell' (cho phép adapt vào nhiều cell types)
- Hungarian matching predicted masks ↔ GT instances theo IoU
- Cross-entropy loss trên matched detections

**Prerequisites:**
- `hipinhththu/pannuke`
- `hipinhththu/sam3-native-pt`
- `phase-a2-lora-weights` (sam3_lora_rank16_final.pt)

**Compute budget:** ~4-5h trên Kaggle T4.

## 00 — Setup

In [ ]:
import subprocess, sys, os, platform, time, json
print("Python  :", sys.version.split()[0])
import torch
print("Torch   :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

In [ ]:
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/sam3_research"
SAM3_DIR = f"{REPO_DIR}/sam3"
CHECKPOINT_PATH = "/kaggle/input/datasets/hipinhththu/sam3-native-pt/sam3.pt"
DATA_ROOT = "/kaggle/input/datasets/hipinhththu/pannuke"

LORA_CKPT_CANDIDATES = [
    "/kaggle/input/datasets/hipinhththu/phase-a2-lora-weights/sam3_lora_rank16_final.pt",
    "/kaggle/input/phase-a2-lora-weights/sam3_lora_rank16_final.pt",
]
LORA_CKPT_PATH = next((p for p in LORA_CKPT_CANDIDATES if os.path.exists(p)), None)
assert LORA_CKPT_PATH, f"Khong tim thay LoRA. Da check: {LORA_CKPT_CANDIDATES}"
print(f"LoRA: {LORA_CKPT_PATH}")

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone",
                    "https://github.com/duonguwu/sam3_research.git", REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

assert os.path.exists(CHECKPOINT_PATH), "Attach hipinhththu/sam3-native-pt"
assert os.path.exists(DATA_ROOT), "Attach hipinhththu/pannuke"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", SAM3_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-image", "scikit-learn", "matplotlib", "opencv-python",
                "pycocotools", "einops", "tqdm"], check=True)
print("OK setup")

## Helper modules (writefile)

In [ ]:
%%writefile pannuke_loader.py
from __future__ import annotations
from pathlib import Path
from typing import Iterator, List, Optional, Tuple
import numpy as np

CELL_TYPES: List[str] = [
    "Neoplastic", "Inflammatory", "Connective", "Dead", "Epithelial",
]

DEFAULT_ROOT = Path("/kaggle/input/datasets/hipinhththu/pannuke")

def _to_uint8(arr: np.ndarray) -> np.ndarray:
    if arr.dtype == np.uint8:
        return arr
    if arr.max() <= 1.0:
        return (arr * 255).round().clip(0, 255).astype(np.uint8)
    return arr.clip(0, 255).astype(np.uint8)

class PanNukeFold:

    def __init__(self, root, fold: int):
        self.root = Path(root)
        self.fold = fold
        sub = f"Fold {fold}"
        f = f"fold{fold}"
        candidates = [
            self.root / f / sub,   
            self.root / sub,        
        ]
        base = next((c for c in candidates if (c / "images" / f / "images.npy").exists()),
                    None)
        if base is None:
            raise FileNotFoundError(
                f"Không tìm thấy Fold {fold} ở: " + " hoặc ".join(str(c) for c in candidates)
            )
        self.images_path = base / "images" / f / "images.npy"
        self.masks_path  = base / "masks"  / f / "masks.npy"

        
        type_candidates = [
            base / "images" / f / "types.npy",   
            base / "types" / f / "types.npy",     
            base / "types.npy",
            self.root / f / "types.npy",
            self.root / "types" / f / "types.npy",
        ]
        self.types_path = next((p for p in type_candidates if p.exists()), None)

        for p in (self.images_path, self.masks_path):
            if not p.exists():
                raise FileNotFoundError(f"Missing: {p}")

        self.images = np.load(self.images_path, mmap_mode="r")
        self.masks  = np.load(self.masks_path,  mmap_mode="r")
        if self.types_path is not None:
            self.tissue_types = np.load(self.types_path, allow_pickle=True)
        else:
            n = self.images.shape[0]
            self.tissue_types = np.array(["unknown"] * n, dtype=object)
            print(f"[Fold {fold}] WARN: types.npy không tìm thấy ở:")
            for c in type_candidates:
                print(f"  - {c}")
            print(f"[Fold {fold}] Dùng placeholder 'unknown' cho {n} ảnh.")

        assert self.images.shape[0] == self.masks.shape[0] == self.tissue_types.shape[0]
        assert self.images.shape[1:] == (256, 256, 3), f"unexpected: {self.images.shape}"
        assert self.masks.shape[1:]  == (256, 256, 6), f"unexpected: {self.masks.shape}"

    def __len__(self) -> int:
        return self.images.shape[0]

    def __getitem__(self, idx: int) -> dict:
        img = _to_uint8(np.array(self.images[idx]))
        m_all = np.array(self.masks[idx], dtype=np.int32)
        masks_per_type = m_all[..., :5].transpose(2, 0, 1)
        counts = np.array(
            [int(np.unique(masks_per_type[k]).size - 1) for k in range(5)],
            dtype=np.int32,
        )
        return {
            "image": img,
            "masks": masks_per_type,
            "counts": counts,
            "tissue": str(self.tissue_types[idx]),
            "fold": self.fold,
            "idx": idx,
        }

    def iter_samples(self, start: int = 0, end: Optional[int] = None) -> Iterator[dict]:
        end = end or len(self)
        for i in range(start, end):
            yield self[i]

def load_all_folds(root=DEFAULT_ROOT) -> Tuple[PanNukeFold, PanNukeFold, PanNukeFold]:
    root = Path(root)
    return PanNukeFold(root, 1), PanNukeFold(root, 2), PanNukeFold(root, 3)

In [ ]:
%%writefile metrics.py
from __future__ import annotations
from typing import Sequence
import numpy as np

def binary_iou(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0

def binary_dice(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    sa, sb = a.sum(), b.sum()
    return float(2 * inter) / float(sa + sb) if (sa + sb) > 0 else 0.0

def match_pred_to_gt(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray],
                     iou_thresh: float = 0.5) -> dict:
    if not pred_masks and not gt_masks:
        return {"tp": 0, "fp": 0, "fn": 0, "mean_iou": 0.0}
    if not pred_masks:
        return {"tp": 0, "fp": 0, "fn": len(gt_masks), "mean_iou": 0.0}
    if not gt_masks:
        return {"tp": 0, "fp": len(pred_masks), "fn": 0, "mean_iou": 0.0}

    iou_matrix = np.zeros((len(pred_masks), len(gt_masks)), dtype=np.float32)
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            iou_matrix[i, j] = binary_iou(pm, gm)

    matched_pred, matched_gt = set(), set()
    ious = []
    pairs = np.dstack(np.unravel_index(np.argsort(-iou_matrix.ravel()), iou_matrix.shape))[0]
    for i, j in pairs:
        if iou_matrix[i, j] < iou_thresh:
            break
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(int(i)); matched_gt.add(int(j))
        ious.append(float(iou_matrix[i, j]))

    tp = len(matched_pred)
    fp = len(pred_masks) - tp
    fn = len(gt_masks)  - len(matched_gt)
    return {"tp": tp, "fp": fp, "fn": fn,
            "mean_iou": float(np.mean(ious)) if ious else 0.0}

def panoptic_quality(stats: dict) -> dict:
    tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
    sq = stats["mean_iou"]
    denom = tp + 0.5 * fp + 0.5 * fn
    rq = tp / denom if denom > 0 else 0.0
    pq = sq * rq
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"PQ": pq, "SQ": sq, "RQ": rq, "F1": f1, "P": precision, "R": recall}

def aggregate_iou_image(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray]) -> float:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu)

def aggregate_iou_dice_image(pred_masks: Sequence[np.ndarray],
                              gt_masks: Sequence[np.ndarray]) -> tuple:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu), binary_dice(pu, gu)

def union_masks(masks: Sequence[np.ndarray], shape=(256, 256)) -> np.ndarray:
    u = np.zeros(shape, dtype=bool)
    for m in masks:
        u |= m.astype(bool)
    return u.astype(np.uint8)

class ClassWiseAccumulator:

    def __init__(self, class_names):
        self.class_names = list(class_names)
        self.tp = {c: 0 for c in self.class_names}
        self.fp = {c: 0 for c in self.class_names}
        self.fn = {c: 0 for c in self.class_names}

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray, class_name: str):
        p = pred_mask.astype(bool)
        g = gt_mask.astype(bool)
        self.tp[class_name] += int(np.logical_and(p, g).sum())
        self.fp[class_name] += int(np.logical_and(p, np.logical_not(g)).sum())
        self.fn[class_name] += int(np.logical_and(np.logical_not(p), g).sum())

    def class_iou(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = tp + fp + fn
        return float(tp) / float(denom) if denom > 0 else 0.0

    def class_dice(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = 2 * tp + fp + fn
        return float(2 * tp) / float(denom) if denom > 0 else 0.0

    def mIoU(self) -> float:
        return float(np.mean([self.class_iou(c) for c in self.class_names]))

    def mDice(self) -> float:
        return float(np.mean([self.class_dice(c) for c in self.class_names]))

    def summary(self) -> dict:
        per_class = {c: {"IoU": self.class_iou(c), "Dice": self.class_dice(c),
                          "TP": self.tp[c], "FP": self.fp[c], "FN": self.fn[c]}
                      for c in self.class_names}
        return {
            "mIoU": self.mIoU(),
            "mDice": self.mDice(),
            "per_class": per_class,
        }

class PerPromptClassAccumulator:

    def __init__(self, class_names, prompts_per_class):
        self.class_names = list(class_names)
        self.prompts_per_class = {c: list(prompts_per_class[c]) for c in self.class_names}
        
        self.accs = {}
        for c, prompts in self.prompts_per_class.items():
            for p in prompts:
                self.accs[(c, p)] = ClassWiseAccumulator([c])

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray,
               class_name: str, prompt: str):
        self.accs[(class_name, prompt)].update(pred_mask, gt_mask, class_name)

    def summary(self) -> dict:
        per_class = {}
        for c in self.class_names:
            per_prompt = []
            for p in self.prompts_per_class[c]:
                acc = self.accs[(c, p)]
                per_prompt.append({
                    "prompt": p,
                    "IoU": acc.class_iou(c),
                    "Dice": acc.class_dice(c),
                    "TP": acc.tp[c], "FP": acc.fp[c], "FN": acc.fn[c],
                })
            ious = [pp["IoU"] for pp in per_prompt]
            dices = [pp["Dice"] for pp in per_prompt]
            per_class[c] = {
                "IoU": float(np.mean(ious)),   
                "Dice": float(np.mean(dices)),
                "per_prompt": per_prompt,
            }
        mIoU = float(np.mean([per_class[c]["IoU"] for c in self.class_names]))
        mDice = float(np.mean([per_class[c]["Dice"] for c in self.class_names]))
        return {"mIoU": mIoU, "mDice": mDice, "per_class": per_class}

def bootstrap_ci(values, n_boot: int = 1000, alpha: float = 0.05, seed: int = 0):
    if len(values) == 0:
        return 0.0, 0.0
    rng = np.random.default_rng(seed)
    vals = np.asarray(values, dtype=np.float64)
    boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    lo = float(np.percentile(boots, 100 * alpha / 2))
    hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return lo, hi

In [ ]:
%%writefile lora_sam3.py
from __future__ import annotations
import math
from typing import Iterable, List, Optional, Set, Tuple
import torch
import torch.nn as nn

class LoRALinear(nn.Module):

    def __init__(self, base: nn.Linear, r: int = 16, alpha: int = 32,
                 dropout: float = 0.05):
        super().__init__()
        self.base = base
        
        for p in self.base.parameters():
            p.requires_grad = False

        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        in_f = base.in_features
        out_f = base.out_features
        self.lora_A = nn.Parameter(torch.zeros(r, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.base(x)
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        return out + lora_out * self.scaling

    @property
    def in_features(self) -> int:
        return self.base.in_features

    @property
    def out_features(self) -> int:
        return self.base.out_features

DEFAULT_LORA_TARGETS: Set[str] = {
    
    
    
    "linear1", "linear2",
}

def inject_lora(
    model: nn.Module,
    target_module_names: Iterable[str] = DEFAULT_LORA_TARGETS,
    r: int = 16,
    alpha: int = 32,
    dropout: float = 0.05,
    path_must_contain: str = "decoder",
    verbose: bool = True,
) -> Tuple[List[str], int]:
    target_set = set(target_module_names)
    replaced: List[str] = []

    for parent_name, parent in list(model.named_modules()):
        
        if path_must_contain and path_must_contain not in parent_name:
            continue
        for attr_name, child in list(parent.named_children()):
            if attr_name in target_set and isinstance(child, nn.Linear):
                lora_layer = LoRALinear(child, r=r, alpha=alpha, dropout=dropout)
                
                base_device = next(child.parameters()).device
                lora_layer.lora_A.data = lora_layer.lora_A.data.to(base_device)
                lora_layer.lora_B.data = lora_layer.lora_B.data.to(base_device)
                setattr(parent, attr_name, lora_layer)
                full_path = f"{parent_name}.{attr_name}" if parent_name else attr_name
                replaced.append(full_path)

    n_lora_params = sum(p.numel() for p in model.parameters()
                        if p.requires_grad)

    if verbose:
        print(f"Injected LoRA into {len(replaced)} modules.")
        print(f"LoRA trainable params: {n_lora_params:,} "
              f"({n_lora_params/1e6:.2f}M)")
        if len(replaced) > 0:
            print("Sample paths:")
            for p in replaced[:5]:
                print(f"  {p}")
            if len(replaced) > 5:
                print(f"  ... ({len(replaced)-5} more)")

    return replaced, n_lora_params

def freeze_non_lora(model: nn.Module) -> Tuple[int, int]:
    n_train = 0
    n_total = 0
    for name, p in model.named_parameters():
        n_total += p.numel()
        if "lora_A" in name or "lora_B" in name:
            p.requires_grad = True
            n_train += p.numel()
        else:
            p.requires_grad = False
    return n_train, n_total

def save_lora_state(model: nn.Module, path: str):
    state = {n: p.detach().cpu()
             for n, p in model.named_parameters()
             if ("lora_A" in n or "lora_B" in n)}
    torch.save(state, path)
    return len(state)

def load_lora_state(model: nn.Module, path: str) -> int:
    state = torch.load(path, map_location="cpu")
    own = dict(model.named_parameters())
    n_loaded = 0
    for k, v in state.items():
        if k in own:
            own[k].data = v.to(own[k].device)
            n_loaded += 1
    return n_loaded

In [ ]:
%%writefile sam3_train.py
from __future__ import annotations
from typing import Dict, List, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import v2

def make_transform(resolution: int = 1008):
    return v2.Compose([
        v2.ToDtype(torch.uint8, scale=True),
        v2.Resize(size=(resolution, resolution)),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])

@torch.no_grad()
def encode_image_frozen(model, transform, pil_img, device: str = "cuda"):
    image = v2.functional.to_image(pil_img).to(device)
    image = transform(image).unsqueeze(0)
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        backbone_out = model.backbone.forward_image(image)

        
        inst_pred = getattr(model, "inst_interactive_predictor", None)
        if inst_pred is not None and "sam2_backbone_out" in backbone_out:
            sam2_bb = backbone_out["sam2_backbone_out"]
            sam2_bb["backbone_fpn"][0] = (
                inst_pred.model.sam_mask_decoder.conv_s0(sam2_bb["backbone_fpn"][0])
            )
            sam2_bb["backbone_fpn"][1] = (
                inst_pred.model.sam_mask_decoder.conv_s1(sam2_bb["backbone_fpn"][1])
            )
    return backbone_out

def encode_text(model, prompt: str, device: str = "cuda"):
    with torch.no_grad():
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            return model.backbone.forward_text([prompt], device=device)

def forward_decoder_with_grad(model, backbone_out: Dict, find_stage,
                              geometric_prompt, device: str = "cuda") -> Dict:
    was_training = model.training
    if was_training:
        model.eval()
    try:
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            outputs = model.forward_grounding(
                backbone_out=backbone_out,
                find_input=find_stage,
                geometric_prompt=geometric_prompt,
                find_target=None,
            )
    finally:
        if was_training:
            model.train()
    return outputs

def semantic_union_mask(outputs: Dict, target_hw: Tuple[int, int]) -> torch.Tensor:
    
    pred_logits = outputs["pred_logits"].float()         
    pred_masks  = outputs["pred_masks"].float()          
    pres_logit  = outputs["presence_logit_dec"].float()  

    
    cls_prob = pred_logits.sigmoid()                   
    pres     = pres_logit.sigmoid().unsqueeze(1)       
    prob     = (cls_prob * pres).squeeze(-1)           

    
    masks = F.interpolate(
        pred_masks, size=target_hw, mode="bilinear", align_corners=False
    ).sigmoid()  

    
    weighted = prob[:, :, None, None] * masks   

    
    
    
    eps = 1e-4
    log_one_minus = torch.log(torch.clamp(1.0 - weighted, min=eps, max=1.0 - eps))
    log_prod = log_one_minus.sum(dim=1)         
    union = 1.0 - torch.exp(torch.clamp(log_prod, min=-20.0))  
    union = torch.clamp(union, min=eps, max=1.0 - eps)  
    return union.squeeze(0)                     

def dice_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    inter = (pred * target).sum()
    return 1.0 - (2.0 * inter + eps) / (pred.sum() + target.sum() + eps)

def bce_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    p = torch.clamp(pred, eps, 1 - eps)
    return -(target * torch.log(p) + (1 - target) * torch.log(1 - p)).mean()

def semantic_seg_loss(pred: torch.Tensor, target: torch.Tensor,
                      bce_weight: float = 0.5,
                      dice_weight: float = 1.0) -> Tuple[torch.Tensor, Dict[str, float]]:
    bce = bce_loss(pred, target)
    dice = dice_loss(pred, target)
    total = bce_weight * bce + dice_weight * dice
    return total, {"bce": float(bce.item()), "dice": float(dice.item()),
                   "loss": float(total.item())}

@torch.no_grad()
def inference_to_binary(outputs: Dict, target_hw: Tuple[int, int],
                        score_threshold: float = 0.3) -> torch.Tensor:
    pred_logits = outputs["pred_logits"]
    pred_masks  = outputs["pred_masks"]
    pres_logit  = outputs["presence_logit_dec"]

    cls_prob = pred_logits.sigmoid()
    pres = pres_logit.sigmoid().unsqueeze(1)
    prob = (cls_prob * pres).squeeze(-1).squeeze(0)  

    masks = F.interpolate(
        pred_masks, size=target_hw, mode="bilinear", align_corners=False
    ).sigmoid().squeeze(0)  

    keep = prob > score_threshold
    if keep.sum() == 0:
        return torch.zeros(target_hw, dtype=torch.bool, device=pred_logits.device)

    selected = (masks[keep] > 0.5)
    union = selected.any(dim=0)
    return union

In [ ]:
%%writefile type_head.py
from __future__ import annotations
from typing import Dict, List, Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class TypeHead(nn.Module):
    def __init__(self, in_dim: int = 256, hidden_dim: int = 128,
                 num_classes: int = 5, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

def roi_pool_feature(backbone_features: torch.Tensor,
                     mask: torch.Tensor) -> torch.Tensor:
    if backbone_features.dim() == 4:
        backbone_features = backbone_features[0]  
    D, Hf, Wf = backbone_features.shape

    
    mask_resized = F.interpolate(
        mask.float()[None, None],
        size=(Hf, Wf),
        mode='bilinear',
        align_corners=False,
    )[0, 0]  

    
    mass = mask_resized.sum()
    if mass < 1e-3:
        return backbone_features.mean(dim=(1, 2))  
    pooled = (backbone_features * mask_resized).sum(dim=(1, 2)) / mass
    return pooled  

def compute_iou_matrix(pred_masks: List[np.ndarray],
                       gt_masks: List[np.ndarray]) -> np.ndarray:
    N_p, N_g = len(pred_masks), len(gt_masks)
    iou_matrix = np.zeros((N_p, N_g), dtype=np.float32)
    for i, p in enumerate(pred_masks):
        for j, g in enumerate(gt_masks):
            inter = np.logical_and(p, g).sum()
            union = np.logical_or(p, g).sum()
            iou_matrix[i, j] = inter / (union + 1e-6) if union > 0 else 0.0
    return iou_matrix

def hungarian_match(iou_matrix: np.ndarray,
                    iou_thresh: float = 0.3) -> List[Tuple[int, int]]:
    try:
        from scipy.optimize import linear_sum_assignment
        
        row_ind, col_ind = linear_sum_assignment(-iou_matrix)
        matches = []
        for r, c in zip(row_ind, col_ind):
            if iou_matrix[r, c] >= iou_thresh:
                matches.append((int(r), int(c)))
        return matches
    except ImportError:
        
        matches = []
        used_g = set()
        
        flat = [(iou_matrix[i, j], i, j)
                for i in range(iou_matrix.shape[0])
                for j in range(iou_matrix.shape[1])]
        flat.sort(reverse=True)
        used_p = set()
        for iou, i, j in flat:
            if iou < iou_thresh:
                break
            if i in used_p or j in used_g:
                continue
            matches.append((i, j))
            used_p.add(i)
            used_g.add(j)
        return matches

def extract_gt_instances(sample: dict, cell_types: List[str]
                         ) -> Tuple[List[np.ndarray], List[int]]:
    masks_per_type = sample["masks"]
    gt_masks = []
    gt_classes = []
    for type_idx in range(5):
        inst_ids = np.unique(masks_per_type[type_idx])
        for inst_id in inst_ids:
            if inst_id == 0:
                continue
            mask = (masks_per_type[type_idx] == inst_id).astype(bool)
            if mask.sum() < 5:  
                continue
            gt_masks.append(mask)
            gt_classes.append(type_idx)
    return gt_masks, gt_classes

def per_class_counts(pred_scores: np.ndarray,
                     pred_probs: np.ndarray) -> np.ndarray:
    counts = (pred_scores[:, None] * pred_probs).sum(axis=0)  
    return counts

def per_class_variance(pred_scores: np.ndarray,
                       pred_probs: np.ndarray) -> np.ndarray:
    weighted = pred_scores[:, None] * pred_probs  
    var = (weighted * (1.0 - weighted)).sum(axis=0)
    return var

In [ ]:
import sys
for p in [".", "/kaggle/working", SAM3_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from pannuke_loader import PanNukeFold, CELL_TYPES, DEFAULT_ROOT
from metrics import ClassWiseAccumulator
from lora_sam3 import (inject_lora, freeze_non_lora, load_lora_state,
                       DEFAULT_LORA_TARGETS)
from sam3_train import (make_transform, encode_image_frozen, encode_text,
                        forward_decoder_with_grad, inference_to_binary)
from type_head import (TypeHead, roi_pool_feature, compute_iou_matrix,
                       hungarian_match, extract_gt_instances,
                       per_class_counts, per_class_variance)
print("Helpers loaded.")

## 01 — Build SAM3 + Inject LoRA + Load A2 weights

In [ ]:
from sam3.model_builder import build_sam3_image_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Build SAM3...")
model = build_sam3_image_model(
    device=device, eval_mode=True,
    checkpoint_path=CHECKPOINT_PATH, load_from_HF=False,
)
model.eval()

LORA_R = 16
LORA_ALPHA = 32
replaced, n_lora = inject_lora(
    model, target_module_names=DEFAULT_LORA_TARGETS,
    r=LORA_R, alpha=LORA_ALPHA, dropout=0.0,
)
n_loaded = load_lora_state(model, LORA_CKPT_PATH)
print(f"LoRA injected {len(replaced)} modules, {n_loaded} tensors loaded")

for p in model.parameters():
    p.requires_grad = False
model.eval()
print("Model fully frozen (SAM3 + LoRA both frozen). Train ONLY TypeHead.")

## 02 — Initialize TypeHead

In [ ]:
type_head = TypeHead(in_dim=256, hidden_dim=128, num_classes=5, dropout=0.1).to(device)
n_type_params = sum(p.numel() for p in type_head.parameters())
print(f"TypeHead params: {n_type_params:,} ({n_type_params/1e3:.1f}K)")
print(f"  Architecture: Linear(256, 128) -> LayerNorm -> ReLU -> Dropout -> Linear(128, 5)")

## 03 — Dataloader

In [ ]:
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

fold1 = PanNukeFold(DEFAULT_ROOT, 1)
fold2 = PanNukeFold(DEFAULT_ROOT, 2)
print(f"Train: Fold 1+2 = {len(fold1)+len(fold2)} patches (Fold 3 NOT loaded)")

TRAIN_PROMPT = "cell"
print(f"Train prompt: '{TRAIN_PROMPT}' (Generic, lay tat ca cells)")

## 04 — Inference pipeline for type head

In [ ]:
from sam3.model.data_misc import FindStage
import torch.nn.functional as F

transform = make_transform(resolution=1008)
find_stage = FindStage(
    img_ids=torch.tensor([0], device=device, dtype=torch.long),
    text_ids=torch.tensor([0], device=device, dtype=torch.long),
    input_boxes=None, input_boxes_mask=None, input_boxes_label=None,
    input_points=None, input_points_mask=None,
)
SCORE_THRESH = 0.3

@torch.no_grad()
def run_sam3_inference(pil_img, prompt):
    """Run SAM3+LoRA va return: (pred_masks, scores, backbone_feat).

    pred_masks: list of (256, 256) bool
    scores: list of float
    backbone_feat: (D, H', W') tensor de roi-pool features
    """
    backbone_out = encode_image_frozen(model, transform, pil_img, device=device)

    feat = None
    if "vision_features" in backbone_out:
        feat = backbone_out["vision_features"]
    elif "backbone_fpn" in backbone_out:
        feat = backbone_out["backbone_fpn"][-1]
    else:
        for k, v in backbone_out.items():
            if isinstance(v, torch.Tensor) and v.dim() == 4:
                feat = v
                break
    assert feat is not None, "Cannot find backbone feature for ROI pooling"
    if feat.dim() == 4:
        feat = feat[0]

    text_out = encode_text(model, prompt, device=device)
    backbone_out.update(text_out)

    geom = model._get_dummy_prompt()
    outputs = forward_decoder_with_grad(
        model, backbone_out, find_stage, geom
    )

    pred_logits = outputs["pred_logits"].float()
    pred_masks  = outputs["pred_masks"].float()
    pres_logit  = outputs["presence_logit_dec"].float()
    cls_prob = pred_logits.sigmoid()
    pres = pres_logit.sigmoid().unsqueeze(1)
    prob = (cls_prob * pres).squeeze(-1).squeeze(0)

    masks_up = F.interpolate(
        pred_masks, size=(256, 256), mode="bilinear", align_corners=False
    ).sigmoid().squeeze(0)
    masks_bin = (masks_up > 0.5)

    keep = prob > SCORE_THRESH
    if keep.sum() == 0:
        return [], [], feat

    pred_masks_list = [masks_bin[i].cpu().numpy().astype(bool)
                       for i in range(len(masks_bin)) if keep[i]]
    scores_list = prob[keep].cpu().tolist()
    return pred_masks_list, scores_list, feat

def predict_types_for_image(pil_img, prompt=TRAIN_PROMPT):
    """Run inference + type head. Return per-detection type probs.

    Returns:
        pred_masks: list of (256,256) bool
        scores: list of float
        type_probs: (N, 5) softmax probs
        features: (N, 256) pooled features
    """
    pred_masks_list, scores_list, backbone_feat = run_sam3_inference(pil_img, prompt)
    N = len(pred_masks_list)
    if N == 0:
        return [], [], np.zeros((0, 5)), np.zeros((0, 256))

    features = torch.zeros(N, 256, device=device)
    for i, pm in enumerate(pred_masks_list):
        mask_t = torch.from_numpy(pm).to(device)
        features[i] = roi_pool_feature(backbone_feat, mask_t)

    with torch.no_grad():
        type_logits = type_head(features)
        type_probs = type_logits.softmax(dim=-1)

    return pred_masks_list, scores_list, type_probs.cpu().numpy(), features.cpu().numpy()

print("Inference pipeline ready.")

## 05 — Training Loop

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

NUM_EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_STEPS = 300
MAX_TRAIN_PER_EPOCH = 2500
IOU_THRESH = 0.3
LOG_EVERY = 100

optimizer = AdamW(type_head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * MAX_TRAIN_PER_EPOCH)

train_indices = list(range(len(fold1))) + [(len(fold1) + i) for i in range(len(fold2))]

def train_step(sample):
    """1 image. Forward + Hungarian match + cross-entropy."""
    pil_img = Image.fromarray(sample["image"]).convert("RGB")
    pred_masks_list, scores_list, backbone_feat = run_sam3_inference(
        pil_img, TRAIN_PROMPT
    )
    if len(pred_masks_list) == 0:
        return None

    gt_masks, gt_classes = extract_gt_instances(sample, CELL_TYPES)
    if len(gt_masks) == 0:
        return None

    iou_matrix = compute_iou_matrix(pred_masks_list, gt_masks)
    matches = hungarian_match(iou_matrix, iou_thresh=IOU_THRESH)
    if len(matches) == 0:
        return None

    matched_features = []
    matched_labels = []
    for pred_i, gt_j in matches:
        mask_t = torch.from_numpy(pred_masks_list[pred_i]).to(device)
        feat = roi_pool_feature(backbone_feat, mask_t)
        matched_features.append(feat)
        matched_labels.append(gt_classes[gt_j])

    features = torch.stack(matched_features)
    labels = torch.tensor(matched_labels, dtype=torch.long, device=device)

    type_logits = type_head(features)
    loss = F.cross_entropy(type_logits, labels)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(type_head.parameters(), max_norm=1.0)
    optimizer.step()

    pred_labels = type_logits.argmax(dim=-1)
    accuracy = (pred_labels == labels).float().mean().item()

    return {"loss": loss.item(), "acc": accuracy, "n_matched": len(matches)}

training_log = {"config": {
    "num_epochs": NUM_EPOCHS, "lr": LEARNING_RATE,
    "max_train_per_epoch": MAX_TRAIN_PER_EPOCH,
    "iou_thresh": IOU_THRESH,
    "lora_ckpt": LORA_CKPT_PATH,
}, "epochs": []}

global_step = 0
t0 = time.time()

all_folds = [fold1, fold2]

for epoch in range(NUM_EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{NUM_EPOCHS} =====")
    type_head.train()
    losses, accs = [], []

    indices = list(range(len(fold1) + len(fold2)))
    random.shuffle(indices)
    indices = indices[:MAX_TRAIN_PER_EPOCH]

    pbar = tqdm(indices, desc=f"Epoch {epoch+1}")
    for step, idx in enumerate(pbar):
        if idx < len(fold1):
            sample = fold1[idx]
        else:
            sample = fold2[idx - len(fold1)]

        if global_step < WARMUP_STEPS:
            lr_now = LEARNING_RATE * (global_step + 1) / WARMUP_STEPS
            for g in optimizer.param_groups:
                g["lr"] = lr_now
        else:
            scheduler.step()

        result = train_step(sample)
        if result is None:
            continue

        losses.append(result["loss"])
        accs.append(result["acc"])
        global_step += 1

        if global_step % LOG_EVERY == 0:
            recent_loss = np.mean(losses[-LOG_EVERY:])
            recent_acc = np.mean(accs[-LOG_EVERY:])
            pbar.set_postfix({
                "loss": f"{recent_loss:.4f}",
                "acc": f"{recent_acc:.3f}",
                "lr": f"{optimizer.param_groups[0]['lr']:.1e}",
            })

    epoch_loss = float(np.mean(losses)) if losses else 0.0
    epoch_acc = float(np.mean(accs)) if accs else 0.0
    elapsed = time.time() - t0
    print(f"  Epoch {epoch+1} done. Avg loss: {epoch_loss:.4f}, Acc: {epoch_acc*100:.2f}%")
    print(f"  Elapsed: {elapsed/60:.1f} min")
    training_log["epochs"].append({
        "epoch": epoch + 1,
        "loss": epoch_loss,
        "acc": epoch_acc,
        "n_steps": len(losses),
        "elapsed_seconds": elapsed,
    })

    ckpt_path = f"{WORK}/type_head_epoch{epoch+1}.pt"
    torch.save(type_head.state_dict(), ckpt_path)
    print(f"  Saved: {ckpt_path}")

final_path = f"{WORK}/type_head_final.pt"
torch.save(type_head.state_dict(), final_path)
print(f"\n===== Training done. Final: {final_path} =====")

## 06 — Save Training Artifacts (NO eval here)

Eval đã tách riêng → `sam3_pannuke_phaseA3_eval.ipynb` trên **full Fold 3**.
Notebook này chỉ train + save TypeHead checkpoints, không leak Fold 3.

In [ ]:
final_results = {
    "config": training_log["config"],
    "training_log": training_log,
    "data_split": "Fold 1+2 train ONLY, Fold 3 held out for eval notebook",
    "checkpoints": {
        "final": f"{WORK}/type_head_final.pt",
        "per_epoch": [f"{WORK}/type_head_epoch{e+1}.pt" for e in range(NUM_EPOCHS)],
    },
}
with open(f"{WORK}/phase_A3_train_log.json", "w") as f:
    json.dump(final_results, f, indent=2)
print(f"Saved: {WORK}/phase_A3_train_log.json")

print("\n" + "=" * 70)
print("PHASE A3 TRAIN DONE — no eval here (strict Fold 3 holdout)")
print("=" * 70)
print(f"  TypeHead weights : {WORK}/type_head_final.pt")
print(f"  Train log JSON   : {WORK}/phase_A3_train_log.json")
print(f"\n  Next steps:")
print(f"    1. Upload type_head_final.pt as Kaggle Dataset")
print(f"    2. Run sam3_pannuke_phaseA3_eval.ipynb on full Fold 3")
print(f"    3. Then Phase C conformal prediction")

## Phase A3 PASS criteria

- **Macro type accuracy ≥ 45%** (vs random 20%)
- **Neoplastic accuracy ≥ 60%** (most common class)
- **Counting MAE per class ≤ 10** cells per image
- **Inflammatory/Connective accuracy ≥ 40%**

**Nếu macro acc < 35%:**
- Tăng hidden_dim TypeHead 128 → 256
- Train thêm epoch (3-5)
- Adjust IOU_THRESH (try 0.2 hoặc 0.5)

**Output cho Phase C:**
- `type_head_final.pt` — TypeHead weights
- `phase_A3_results.json` — per-class accuracy + MAE
- Per-detection $p_{ik}$ ready for Poisson-Binomial conformal counting